## Imports

In [1]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from pathlib import Path
import numpy as np
import json

## Define Project Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed" / "chunks" / "chunks.jsonl"
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"
QDRANT_PATH = PROJECT_ROOT / "data" / "processed" / "vector_store" / "qdrant"
COLLECTION_NAME = "financial_policies"

print("Project root:", PROJECT_ROOT)
print("Chunks file:", CHUNKS_FILE)
print("Qdrant path:", QDRANT_PATH)
print("Collection:", COLLECTION_NAME)

Project root: /Users/pushkarkamat/Desktop/financial-rag
Chunks file: /Users/pushkarkamat/Desktop/financial-rag/data/processed/chunks/chunks.jsonl
Qdrant path: /Users/pushkarkamat/Desktop/financial-rag/data/processed/vector_store/qdrant
Collection: financial_policies


In [4]:
client = QdrantClient(path=str(QDRANT_PATH))
print("Qdrant Client Connected")

Qdrant Client Connected


## Verify the collection

In [4]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Vectors stored:", collection_info.points_count)

Collection: financial_policies
Vectors stored: 295


## Load BGE-large

In [3]:
EMBED_MODEL = "BAAI/bge-large-en-v1.5"
embedding_model = SentenceTransformer(EMBED_MODEL)

print("Embedding model loaded.")
print("Embedding dimension:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 1024


## Create first query

In [6]:
query = "What is the maximum amount that requires enhanced credit approval?"
print("Query:", query)

Query: What is the maximum amount that requires enhanced credit approval?


## Convert Query into Embeddings

In [7]:
embedded_query = embedding_model.encode(
    query,
    normalize_embeddings=True
)

print("Query embedding shape:", embedded_query.shape)
print("Query embedding dtype:", embedded_query.dtype)
print("Query embedding norm:", np.linalg.norm(embedded_query))

Query embedding shape: (1024,)
Query embedding dtype: float32
Query embedding norm: 1.0


## Retrieve the most relevant chunks

In [9]:
search_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=embedded_query.tolist(),
    limit=5,
    with_payload=True,
).points

print("Retrieved chunks:", len(search_results))

Retrieved chunks: 5


## Inspect what Qdrant retrieved

In [10]:
for i, result in enumerate(search_results, start=1):
    print("=" * 75)
    print(f"Result: {i}")
    print(f"Score: {result.score}")
    print(f"Chunk ID: {result.payload.get('chunk_id')}")
    print(f"Document: {result.payload.get('document')}")
    print("Text:")
    print(result.payload.get("text"))
    print()

Result: 1
Score: 0.6610575981423698
Chunk ID: chunk_00000043
Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
Text:
Section: 13. Credit Approval and Delegated Authority

Credit decisions must be made by an officer or committee holding sufficient delegated authority for the product, amount, risk grade and exception status. Authority is personal to the designated role and remains subject to restrictions stated in CDAS-007.

Section: 13. Credit Approval and Delegated Authority

Credit Risk expects documented judgement, clear ownership and evidence that can be reconstructed after the decision. The requirements in this section apply unless a documented product rule, approved exception, or later effective instruction expressly provides otherwise.

Section: 13. Credit Approval and Delegated Authority

Control requirements

Section: 13. Credit Approval and Delegated Authority

• CA1 may approve standard personal loans up to ₹10 lakh but is limited to ₹8 lakh for new-vehicle finance, ₹6 lakh f

----

## Testing Queries

In [11]:
test_queries = [
    "What is the maximum personal loan amount that CA2 can approve?",
    "How much can CA2 authorize for a standard personal loan?",
    "What is Northstar's maximum credit-card limit for students?"
]

for query in test_queries:
    print("=" * 80)
    print("QUERY:", query)

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=3,
        with_payload=True,
    ).points

    for i, result in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Score:", round(result.score, 4))
        print("Chunk:", result.payload.get("chunk_id"))
        print("Document:", result.payload.get("document"))
        print("Text:", result.payload.get("text")[:500])

QUERY: What is the maximum personal loan amount that CA2 can approve?

Result 1
Score: 0.7522
Chunk: chunk_00000159
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx
Text: Section: 4. Personal Loan Authority

Standard personal loans retain the general CA1 and CA2 retail limits but remain subject to the product cap and any elevated authority triggered by risk or exception status.

Section: 4. Personal Loan Authority

Authority administration treats the controlled schedule as the source of decision rights; workflow permissions are supporting evidence only. The requirements in this section apply unless a documented product rule, approved exception, or later effective

Result 2
Score: 0.7426
Chunk: chunk_00000105
Document: 02_Credit_Exception_Procedure_CEP-006_v1.3.docx
Text: • A personal-loan score of 620–639 is an E2 exception and requires at least CA3 approval in addition to manual underwriting.

Section: 7. Credit-Score and Thin-File Exceptions

• A score below 620 is

---

## def retrieval

In [12]:
def retrieve_documents(query, top_k=5):
    """
    Retrieve the most relevant policy chunks from Qdrant.
    """

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=top_k,
        with_payload=True,
    ).points

    return results

### testing 

In [13]:
results = retrieve_documents(
    "How much can CA2 authorize for a standard personal loan?",
    top_k=5
)

for i, result in enumerate(results, start=1):
    print("=" * 70)
    print(f"Result {i}")
    print("Score:", round(result.score, 4))
    print("Chunk:", result.payload.get("chunk_id"))
    print("Document:", result.payload.get("document"))
    print()

Result 1
Score: 0.7696
Chunk: chunk_00000159
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx

Result 2
Score: 0.7533
Chunk: chunk_00000105
Document: 02_Credit_Exception_Procedure_CEP-006_v1.3.docx

Result 3
Score: 0.7447
Chunk: chunk_00000199
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx

Result 4
Score: 0.7338
Chunk: chunk_00000043
Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx

Result 5
Score: 0.7262
Chunk: chunk_00000172
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx



---

In [5]:
query = "What are the requirements for credit approval?"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

print("Query:", query)
print("Embedding shape:", query_embedding.shape)
print("Embedding norm:", np.linalg.norm(query_embedding))

Query: What are the requirements for credit approval?
Embedding shape: (1024,)
Embedding norm: 1.0


In [6]:
search_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding.tolist(),
    limit=5,
    with_payload=True,
).points

print("Retrieved chunks:", len(search_results))

Retrieved chunks: 5


In [7]:
for i, result in enumerate(search_results):
    print("=" * 70)
    print(f"Result {i + 1}")
    print("Score:", result.score)
    print("Document:", result.payload.get("document"))
    print("Page:", result.payload.get("page"))
    print("Section:", result.payload.get("section"))
    print("Text:")
    print(result.payload.get("text"))

Result 1
Score: 0.7251141416185554
Document: rag_demo_test.pdf
Page: 1
Section: None
Text:
RAG Demo Test

Loan approval requires verified income, identity documents, and acceptable credit history.
Result 2
Score: 0.7127141017898487
Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
Page: None
Section: 7. Credit Bureau and Credit Score Requirements
Text:
Section: 7. Credit Bureau and Credit Score Requirements

• Absence of a score or a thin-file result requires alternative evidence and manual review where the product permits such customers.

Section: 7. Credit Bureau and Credit Score Requirements

• Bureau data shall be matched to the applicant using approved identity controls and material discrepancies investigated.

Section: 7. Credit Bureau and Credit Score Requirements

• A score obtained outside the product's permitted freshness period shall be refreshed before final decision.
Result 3
Score: 0.7091260888217126
Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
Page: None
Section: 13

In [8]:
retrieved_chunks = []

for result in search_results:
    retrieved_chunks.append({
        "score": result.score,
        "text": result.payload.get("text"),
        "document": result.payload.get("document"),
        "page": result.payload.get("page"),
        "section": result.payload.get("section"),
    })

print("Prepared retrieved chunks:", len(retrieved_chunks))

Prepared retrieved chunks: 5


In [9]:
for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i + 1}] {chunk['document']} | "
          f"Page: {chunk['page']} | "
          f"Section: {chunk['section']} | "
          f"Score: {chunk['score']:.4f}")

[1] rag_demo_test.pdf | Page: 1 | Section: None | Score: 0.7251
[2] 01_Credit_Risk_Policy_CRP-001_v2.0.docx | Page: None | Section: 7. Credit Bureau and Credit Score Requirements | Score: 0.7127
[3] 01_Credit_Risk_Policy_CRP-001_v2.0.docx | Page: None | Section: 13. Credit Approval and Delegated Authority | Score: 0.7091
[4] 01_Credit_Risk_Policy_CRP-001_v2.0.docx | Page: None | Section: 13. Credit Approval and Delegated Authority | Score: 0.7051
[5] 04_Loan_Origination_Policy_LOP-002_v2.3.docx | Page: None | Section: 15. Approval and Conditions | Score: 0.7036


In [10]:
def retrieve_chunks(query: str, top_k: int = 5):
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=top_k,
        with_payload=True,
    ).points

    return [
        {
            "score": result.score,
            "text": result.payload.get("text"),
            "document": result.payload.get("document"),
            "page": result.payload.get("page"),
            "section": result.payload.get("section"),
        }
        for result in results
    ]

print("Retrieval function ready.")

Retrieval function ready.


In [11]:
test_results = retrieve_chunks(
    "What is the maximum personal loan amount that CA2 can approve?",
    top_k=3
)

for i, chunk in enumerate(test_results):
    print("=" * 70)
    print(f"Result {i + 1}")
    print("Score:", round(chunk["score"], 4))
    print("Document:", chunk["document"])
    print("Section:", chunk["section"])
    print("Text:")
    print(chunk["text"])

Result 1
Score: 0.7522
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx
Section: 4. Personal Loan Authority
Text:
Section: 4. Personal Loan Authority

Standard personal loans retain the general CA1 and CA2 retail limits but remain subject to the product cap and any elevated authority triggered by risk or exception status.

Section: 4. Personal Loan Authority

Authority administration treats the controlled schedule as the source of decision rights; workflow permissions are supporting evidence only. The requirements in this section apply unless a documented product rule, approved exception, or later effective instruction expressly provides otherwise.

Section: 4. Personal Loan Authority

Control requirements

Section: 4. Personal Loan Authority

• CA1 may approve standard personal loans up to ₹10 lakh.

Section: 4. Personal Loan Authority

• CA2 may approve standard personal loans up to the standard product cap of ₹20 lakh.

Section: 4. Personal Loan Authority

• CA3 a

In [12]:
def build_context(chunks):
    context_parts = []

    for i, chunk in enumerate(chunks, start=1):
        source = chunk["document"]

        if chunk["section"]:
            source += f" | Section: {chunk['section']}"

        if chunk["page"]:
            source += f" | Page: {chunk['page']}"

        context_parts.append(
            f"[Source {i}: {source}]\n"
            f"{chunk['text']}"
        )

    return "\n\n".join(context_parts)


context = build_context(test_results)

print(context)

[Source 1: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 4. Personal Loan Authority]
Section: 4. Personal Loan Authority

Standard personal loans retain the general CA1 and CA2 retail limits but remain subject to the product cap and any elevated authority triggered by risk or exception status.

Section: 4. Personal Loan Authority

Authority administration treats the controlled schedule as the source of decision rights; workflow permissions are supporting evidence only. The requirements in this section apply unless a documented product rule, approved exception, or later effective instruction expressly provides otherwise.

Section: 4. Personal Loan Authority

Control requirements

Section: 4. Personal Loan Authority

• CA1 may approve standard personal loans up to ₹10 lakh.

Section: 4. Personal Loan Authority

• CA2 may approve standard personal loans up to the standard product cap of ₹20 lakh.

Section: 4. Personal Loan Authority

• CA3 and above may approve with

In [13]:
from financial_rag.llm.router import generate_response

prompt = f"""
Answer the question using only the provided context.

Question:
What is the maximum personal loan amount that CA2 can approve?

Context:
{context}

Give a concise answer and mention the source document and section.
"""

response, provider = generate_response(prompt)

print("Provider:", provider)
print("\nAnswer:")
print(response)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Provider: gemini

Answer:
CA2 may approve standard personal loans up to the standard product cap of ₹20 lakh.

Source: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 4. Personal Loan Authority


In [14]:
question = "What happens when a personal-loan applicant has a credit score of 635?"

test_results = retrieve_chunks(question, top_k=5)
context = build_context(test_results)

prompt = f"""
Answer the question using only the provided context.

Question:
{question}

Context:
{context}

Give a concise answer and mention the relevant source document and section.
"""

response, provider = generate_response(prompt)

print("Provider:", provider)
print("\nAnswer:")
print(response)

Provider: gemini

Answer:
When a personal-loan applicant has a credit score of 635, it is classified as an E2 exception and requires manual underwriting in addition to at least CA3 approval.

**Source:** 02_Credit_Exception_Procedure_CEP-006_v1.3.docx, Section 7. Credit-Score and Thin-File Exceptions (and 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx, Section 8. Exception Authority).


In [15]:
def generate_rag_answer(question: str, top_k: int = 5):
    retrieved = retrieve_chunks(question, top_k=top_k)
    context = build_context(retrieved)

    prompt = f"""
Answer the question using only the provided context.

If the context does not contain enough information to answer the question,
say that the information is not available in the provided documents.

Question:
{question}

Context:
{context}

Give a concise answer and cite the relevant source document and section.
"""

    response, provider = generate_response(prompt)

    return {
        "answer": response,
        "provider": provider,
        "sources": retrieved,
    }


print("RAG answer function ready.")

RAG answer function ready.


In [16]:
result = generate_rag_answer(
    "What is the maximum personal loan amount that CA2 can approve?"
)

print("Provider:", result["provider"])
print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for i, source in enumerate(result["sources"], start=1):
    print(
        f"{i}. {source['document']} | "
        f"Section: {source['section']} | "
        f"Score: {source['score']:.4f}"
    )

Provider: gemini

Answer:
CA2 may approve standard personal loans up to the standard product cap of ₹20 lakh.

Source: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 4. Personal Loan Authority

Sources:
1. 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 4. Personal Loan Authority | Score: 0.7522
2. 02_Credit_Exception_Procedure_CEP-006_v1.3.docx | Section: 7. Credit-Score and Thin-File Exceptions | Score: 0.7426
3. 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: Appendix C — Approval Examples | Score: 0.7157
4. 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 8. Exception Authority | Score: 0.7088
5. 01_Credit_Risk_Policy_CRP-001_v2.0.docx | Section: 13. Credit Approval and Delegated Authority | Score: 0.7046


In [17]:
result = generate_rag_answer(
    "What happens when a personal-loan applicant has a credit score of 635?"
)

print("Provider:", result["provider"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for i, source in enumerate(result["sources"], start=1):
    print(
        f"{i}. {source['document']} | "
        f"Section: {source['section']} | "
        f"Score: {source['score']:.4f}"
    )

Provider: gemini

Answer:
When a personal-loan applicant has a credit score of 635, it is classified as an E2 exception and requires manual underwriting in addition to at least CA3 approval.

Source: 02_Credit_Exception_Procedure_CEP-006_v1.3.docx | Section: 7. Credit-Score and Thin-File Exceptions; 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 8. Exception Authority

Sources:
1. 02_Credit_Exception_Procedure_CEP-006_v1.3.docx | Section: 7. Credit-Score and Thin-File Exceptions | Score: 0.7569
2. 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 8. Exception Authority | Score: 0.6844
3. 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 4. Personal Loan Authority | Score: 0.6650
4. 04_Loan_Origination_Policy_LOP-002_v2.3.docx | Section: 15. Approval and Conditions | Score: 0.6442
5. 02_Credit_Exception_Procedure_CEP-006_v1.3.docx | Section: 7. Credit-Score and Thin-File Exceptions | Score: 0.6432


In [18]:
result = generate_rag_answer(
    "What is Northstar Financial's policy for cryptocurrency loans?"
)

print("Provider:", result["provider"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for i, source in enumerate(result["sources"], start=1):
    print(
        f"{i}. {source['document']} | "
        f"Section: {source['section']} | "
        f"Score: {source['score']:.4f}"
    )

Provider: gemini

Answer:
The information is not available in the provided documents.

Sources:
1. 04_Loan_Origination_Policy_LOP-002_v2.3.docx | Section: None | Score: 0.7074
2. 01_Credit_Risk_Policy_CRP-001_v2.0.docx | Section: 2. Scope and Applicability | Score: 0.6570
3. 01_Credit_Risk_Policy_CRP-001_v2.0.docx | Section: None | Score: 0.6509
4. 01_Credit_Risk_Policy_CRP-001_v2.0.docx | Section: 1. Purpose and Policy Objectives | Score: 0.6451
5. 02_Credit_Exception_Procedure_CEP-006_v1.3.docx | Section: None | Score: 0.6444


In [19]:
result = generate_rag_answer(
    "What should happen when a personal-loan applicant has a credit score of 635 and requests a loan of ₹15 lakh?"
)

print("Provider:", result["provider"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for i, source in enumerate(result["sources"], start=1):
    print(
        f"{i}. {source['document']} | "
        f"Section: {source['section']} | "
        f"Score: {source['score']:.4f}"
    )

Provider: gemini

Answer:
For a personal-loan applicant with a credit score of 635 requesting ₹15 lakh, the following must occur:

*   **Exception Handling:** The application is classified as an E2 exception, which requires manual underwriting and at least CA3 approval (Source 1, Section 7; Source 3, Section 8).
*   **Authority Requirement:** Because the application involves a low-score exception, it may require a higher authority than the amount alone (Source 4, Section 4). The decision must be made by an officer holding sufficient delegated authority for the product, amount, and exception status (Source 5, Section 13).

Sources:
1. 02_Credit_Exception_Procedure_CEP-006_v1.3.docx | Section: 7. Credit-Score and Thin-File Exceptions | Score: 0.6940
2. 04_Loan_Origination_Policy_LOP-002_v2.3.docx | Section: 15. Approval and Conditions | Score: 0.6484
3. 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx | Section: 8. Exception Authority | Score: 0.6458
4. 03_Credit_Delegated_Autho

In [20]:
client.close()